# Task-Aware Multimodal Ultrasonic Baseline

Fresh PyTorch notebook for RGB image + ultrasonic FLAC reflection data. The architecture uses separate residual CNN encoders, feature-level fusion, and task-aware heads. It uses the same competition score as the other notebooks and standard multi-task losses: cross entropy for classification and SmoothL1/MSE for distance.

## 1. Imports and Environment

In [ ]:
from __future__ import annotations

import math
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score, mean_squared_error, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch import nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def get_device() -> torch.device:
    print('PyTorch:', torch.__version__)
    print('CUDA:', torch.version.cuda)
    if not torch.cuda.is_available():
        print('CUDA unavailable; using CPU')
        return torch.device('cpu')
    print('GPU:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    try:
        _ = (torch.ones(1, device='cuda') + 1).item()
        return torch.device('cuda')
    except Exception as exc:
        print('CUDA visible but unusable with this PyTorch build:', repr(exc))
        return torch.device('cpu')


## 2. Configuration

Update paths for Kaggle. For ultrasonic data, keep `AUDIO_TARGET_SAMPLE_RATE=None` to avoid downsampling and losing high-frequency content. The model can use log-STFT or log-Mel cached tensors; log-STFT is the default because ultrasonic reflection structure is not human-hearing-like.

In [ ]:
AudioRepresentation = Literal['log_stft', 'log_mel']
FusionType = Literal['concat', 'gated']
DistanceLoss = Literal['smooth_l1', 'mse']

@dataclass
class Config:
    SEED: int = 42
    TRAIN_CSV_PATH: Path = Path('/kaggle/input/your-dataset/train.csv')
    VAL_CSV_PATH: Path | None = Path('/kaggle/input/your-dataset/val.csv')
    TEST_CSV_PATH: Path = Path('/kaggle/input/your-dataset/test.csv')
    IMAGE_DIR: Path = Path('/kaggle/input/your-dataset/images')
    AUDIO_DIR: Path = Path('/kaggle/input/your-dataset/audio')
    SPEC_DIR: Path = Path('/kaggle/working/ultrasonic_spec')
    CHECKPOINT_DIR: Path = Path('/kaggle/working/task_aware_checkpoints')
    SUBMISSION_PATH: Path = Path('/kaggle/working/submission.csv')
    IMAGE_COL: str | None = None
    AUDIO_COL: str | None = None
    OBJECT_COL: str | None = None
    DISTANCE_COL: str | None = None
    ZONE_COL: str | None = None
    ILLUMINATION_COL: str | None = None
    PRECOMPUTE_SPECTROGRAMS: bool = True
    OVERWRITE_SPECTROGRAMS: bool = False
    AUDIO_REPRESENTATION: AudioRepresentation = 'log_stft'
    AUDIO_TARGET_SAMPLE_RATE: int | None = None
    N_FFT: int = 2048
    WIN_LENGTH: int | None = None
    HOP_LENGTH: int = 512
    N_MELS: int = 128
    F_MIN: float = 0.0
    F_MAX: float | None = None
    SPEC_TIME_FRAMES: int | None = None
    SPEC_FREQ_BINS: int | None = None
    USE_FULL_AUDIO: bool = True
    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 16
    GRADIENT_ACCUMULATION_STEPS: int = 2
    NUM_WORKERS: int = 2
    EPOCHS: int = 60
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    WARMUP_EPOCHS: int = 5
    DROPOUT: float = 0.35
    IMAGE_EMBED_DIM: int = 256
    AUDIO_EMBED_DIM: int = 256
    FUSION_DIM: int = 256
    FUSION_TYPE: FusionType = 'gated'
    DISTANCE_LOSS: DistanceLoss = 'smooth_l1'
    USE_DISTANCE_NORMALIZATION: bool = True
    USE_CLASS_WEIGHTS: bool = False
    USE_AMP: bool = True
    GRAD_CLIP_NORM: float = 1.0
    EARLY_STOPPING_PATIENCE: int = 10
    USE_IMAGE: bool = True
    USE_AUDIO: bool = True
    HORIZONTAL_FLIP: bool = True

CFG = Config()
seed_everything(CFG.SEED)
DEVICE = get_device()
CFG.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CFG.SPEC_DIR.mkdir(parents=True, exist_ok=True)
print(CFG)


## 3. Dataset Inspection and Column Detection

In [ ]:
def read_csv_checked(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df

def infer_column(df: pd.DataFrame, candidates: list[str], explicit: str | None = None) -> str:
    if explicit is not None:
        if explicit not in df.columns:
            raise KeyError(f'{explicit} not in columns: {list(df.columns)}')
        return explicit
    lookup = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lookup:
            return lookup[cand.lower()]
    raise KeyError(f'Cannot infer {candidates}. Available: {list(df.columns)}')

train_df_full = read_csv_checked(CFG.TRAIN_CSV_PATH)
provided_val_df = read_csv_checked(CFG.VAL_CSV_PATH) if CFG.VAL_CSV_PATH is not None and CFG.VAL_CSV_PATH.exists() else pd.DataFrame()
test_df = read_csv_checked(CFG.TEST_CSV_PATH) if CFG.TEST_CSV_PATH.exists() else pd.DataFrame()

IMAGE_COL = infer_column(train_df_full, ['image name', 'image_id', 'image', 'filename'], CFG.IMAGE_COL)
AUDIO_COL = infer_column(train_df_full, ['audio name', 'audio_id', 'audio', 'flac', 'mel', 'mel_path'], CFG.AUDIO_COL)
OBJECT_COL = infer_column(train_df_full, ['Object_Type', 'object_type', 'object'], CFG.OBJECT_COL)
DISTANCE_COL = infer_column(train_df_full, ['distance', 'Distance'], CFG.DISTANCE_COL)
ZONE_COL = infer_column(train_df_full, ['Location_Zone', 'location_zone', 'zone'], CFG.ZONE_COL)
ILLUM_COL = infer_column(train_df_full, ['Illumination', 'illumination'], CFG.ILLUMINATION_COL)

print('Detected:', {'image': IMAGE_COL, 'audio': AUDIO_COL, 'object': OBJECT_COL, 'distance': DISTANCE_COL, 'zone': ZONE_COL, 'illumination': ILLUM_COL})
print('Train rows:', len(train_df_full), 'Val rows:', len(provided_val_df), 'Test rows:', len(test_df))
display(train_df_full.head())
display(train_df_full.isna().sum().to_frame('missing'))
target_cols = [OBJECT_COL, DISTANCE_COL, ZONE_COL, ILLUM_COL]
if train_df_full[target_cols].isna().any().any():
    display(train_df_full[train_df_full[target_cols].isna().any(axis=1)].head())
    raise ValueError('Missing training targets')
if len(provided_val_df) and provided_val_df[target_cols].isna().any().any():
    display(provided_val_df[provided_val_df[target_cols].isna().any(axis=1)].head())
    raise ValueError('Missing validation targets')
for col in [OBJECT_COL, ZONE_COL, ILLUM_COL]:
    print('\n', col)
    display(train_df_full[col].value_counts().to_frame('count'))
display(pd.to_numeric(train_df_full[DISTANCE_COL]).describe().to_frame('distance'))


## 4. Full Ultrasonic Spectrogram Cache

This cell uses the full FLAC signal by default. It does not crop raw audio. The model later pads spectrograms to a fixed length for batching. Use `log_stft` for ultrasonic reflections unless you intentionally want `log_mel`.

In [ ]:
def audio_path(value: Any) -> Path:
    p = Path(str(value))
    if p.is_absolute():
        return p
    return CFG.AUDIO_DIR / (p if p.suffix else p.with_suffix('.flac'))

def spec_path(value: Any) -> Path:
    p = Path(str(value))
    if p.suffix.lower() in ['.pt', '.npy']:
        return p if p.is_absolute() else CFG.SPEC_DIR / p
    return (CFG.SPEC_DIR / p).with_suffix('.pt')

def compute_full_spectrogram(path: Path) -> dict[str, Any]:
    waveform, sr = torchaudio.load(str(path))
    original_sr = sr
    waveform = waveform.mean(dim=0, keepdim=True).float()
    if CFG.AUDIO_TARGET_SAMPLE_RATE is not None and CFG.AUDIO_TARGET_SAMPLE_RATE != sr:
        if CFG.AUDIO_TARGET_SAMPLE_RATE < sr:
            print('Warning: downsampling ultrasonic signal:', path.name)
        waveform = torchaudio.functional.resample(waveform, sr, CFG.AUDIO_TARGET_SAMPLE_RATE)
        sr = CFG.AUDIO_TARGET_SAMPLE_RATE
    if CFG.AUDIO_REPRESENTATION == 'log_stft':
        spec = torchaudio.transforms.Spectrogram(n_fft=CFG.N_FFT, win_length=CFG.WIN_LENGTH, hop_length=CFG.HOP_LENGTH, power=2.0)(waveform)
        spec = torch.log1p(spec)
    elif CFG.AUDIO_REPRESENTATION == 'log_mel':
        spec = torchaudio.transforms.MelSpectrogram(sample_rate=sr, n_fft=CFG.N_FFT, win_length=CFG.WIN_LENGTH, hop_length=CFG.HOP_LENGTH, n_mels=CFG.N_MELS, f_min=CFG.F_MIN, f_max=CFG.F_MAX, power=2.0)(waveform)
        spec = torchaudio.transforms.AmplitudeToDB(stype='power')(spec)
    else:
        raise ValueError(CFG.AUDIO_REPRESENTATION)
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    return {'spec': spec.contiguous().cpu().float(), 'source': str(path), 'sample_rate': sr, 'original_sample_rate': original_sr, 'representation': CFG.AUDIO_REPRESENTATION, 'n_fft': CFG.N_FFT, 'hop_length': CFG.HOP_LENGTH}

def precompute_specs(df_list: list[pd.DataFrame]) -> None:
    values = sorted(set(pd.concat([df[[AUDIO_COL]] for df in df_list if len(df)], ignore_index=True)[AUDIO_COL].astype(str)))
    created = skipped = 0
    errors: list[dict[str, str]] = []
    for value in tqdm(values, desc='precompute full ultrasonic spectrogram'):
        out = spec_path(value)
        if out.exists() and not CFG.OVERWRITE_SPECTROGRAMS:
            skipped += 1
            continue
        src = audio_path(value)
        if not src.exists():
            errors.append({'audio': str(src), 'error': 'missing file'})
            continue
        try:
            out.parent.mkdir(parents=True, exist_ok=True)
            torch.save(compute_full_spectrogram(src), out)
            created += 1
        except Exception as exc:
            errors.append({'audio': str(src), 'error': repr(exc)})
    print({'created': created, 'skipped': skipped, 'errors': len(errors), 'spec_dir': str(CFG.SPEC_DIR)})
    if errors:
        display(pd.DataFrame(errors).head(20))
        raise RuntimeError('Spectrogram precompute failed')

if CFG.PRECOMPUTE_SPECTROGRAMS:
    precompute_specs([train_df_full, provided_val_df, test_df])


## 5. Encoders, Split, and Dataset

In [ ]:
def make_encoder(series: pd.Series) -> tuple[dict[str, int], dict[int, str]]:
    classes = sorted(series.astype(str).unique())
    c2i = {c: i for i, c in enumerate(classes)}
    i2c = {i: c for c, i in c2i.items()}
    return c2i, i2c

object_to_idx, idx_to_object = make_encoder(train_df_full[OBJECT_COL])
zone_to_idx, idx_to_zone = make_encoder(train_df_full[ZONE_COL])
illum_to_idx, idx_to_illum = make_encoder(train_df_full[ILLUM_COL])

def split_data(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    combined = df[OBJECT_COL].astype(str) + '__' + df[ZONE_COL].astype(str) + '__' + df[ILLUM_COL].astype(str)
    for name, y in [('combined', combined), ('object', df[OBJECT_COL].astype(str))]:
        try:
            tr, va = train_test_split(df, test_size=0.2, random_state=CFG.SEED, stratify=y)
            print('split:', name)
            return tr.reset_index(drop=True), va.reset_index(drop=True)
        except ValueError as exc:
            print('split fallback from', name, exc)
    tr, va = train_test_split(df, test_size=0.2, random_state=CFG.SEED)
    return tr.reset_index(drop=True), va.reset_index(drop=True)

if len(provided_val_df):
    train_df = train_df_full.reset_index(drop=True)
    valid_df = provided_val_df.reset_index(drop=True)
    print('Using provided validation CSV:', CFG.VAL_CSV_PATH)
else:
    train_df, valid_df = split_data(train_df_full)
dist_mean = float(pd.to_numeric(train_df[DISTANCE_COL]).mean())
dist_std = float(pd.to_numeric(train_df[DISTANCE_COL]).std()) or 1.0
print({'train': len(train_df), 'valid': len(valid_df), 'dist_mean': dist_mean, 'dist_std': dist_std})

RGB_MEAN = [0.485, 0.456, 0.406]
RGB_STD = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.RandomResizedCrop(CFG.IMAGE_SIZE, scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(p=0.5 if CFG.HORIZONTAL_FLIP else 0), transforms.RandomRotation(5), transforms.ColorJitter(0.06, 0.06, 0.04, 0.01), transforms.ToTensor(), transforms.Normalize(RGB_MEAN, RGB_STD)])
valid_tf = transforms.Compose([transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(RGB_MEAN, RGB_STD)])

def image_path(value: Any) -> Path:
    p = Path(str(value))
    return p if p.is_absolute() else CFG.IMAGE_DIR / p

def safe_load(path: Path) -> Any:
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')

def load_spec(path: Path) -> torch.Tensor:
    if not path.exists():
        raise FileNotFoundError(path)
    payload = safe_load(path) if path.suffix == '.pt' else np.load(path)
    spec = payload['spec'] if isinstance(payload, dict) and 'spec' in payload else payload
    spec = torch.as_tensor(spec).float()
    if spec.ndim == 2:
        spec = spec.unsqueeze(0)
    if spec.ndim != 3 or spec.shape[0] != 1:
        raise ValueError(f'bad spec shape {tuple(spec.shape)} for {path}')
    if not torch.isfinite(spec).all():
        raise ValueError(f'NaN/Inf in {path}')
    return spec

def infer_spec_shape(df: pd.DataFrame) -> tuple[int, int]:
    freq, time = [], []
    scan_sources = [train_df_full[[AUDIO_COL]]]
    if len(provided_val_df):
        scan_sources.append(provided_val_df[[AUDIO_COL]])
    if len(test_df):
        scan_sources.append(test_df[[AUDIO_COL]])
    scan = pd.concat(scan_sources, ignore_index=True)
    for value in scan[AUDIO_COL].astype(str):
        s = load_spec(spec_path(value))
        freq.append(int(s.shape[1]))
        time.append(int(s.shape[2]))
    f = CFG.SPEC_FREQ_BINS or int(pd.Series(freq).mode().iloc[0])
    t = CFG.SPEC_TIME_FRAMES or int(max(time))
    print({'spec_freq_bins': f, 'spec_time_frames': t})
    return f, t

SPEC_FREQ_BINS, SPEC_TIME_FRAMES = infer_spec_shape(train_df_full)

def fix_spec(spec: torch.Tensor) -> torch.Tensor:
    if spec.shape[1] != SPEC_FREQ_BINS:
        spec = F.interpolate(spec.unsqueeze(0), size=(SPEC_FREQ_BINS, spec.shape[2]), mode='bilinear', align_corners=False).squeeze(0)
    if spec.shape[2] > SPEC_TIME_FRAMES:
        raise ValueError(f'spec longer than fixed frame length: {spec.shape[2]} > {SPEC_TIME_FRAMES}')
    if spec.shape[2] < SPEC_TIME_FRAMES:
        spec = F.pad(spec, (0, SPEC_TIME_FRAMES - spec.shape[2]))
    return (spec - spec.mean()) / (spec.std() + 1e-6)

class UltrasonicDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tfm: transforms.Compose, has_labels: bool) -> None:
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.has_labels = has_labels
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.df.iloc[idx]
        img = Image.open(image_path(row[IMAGE_COL])).convert('RGB')
        item = {'image': self.tfm(img), 'spec': fix_spec(load_spec(spec_path(row[AUDIO_COL]))), 'sample_id': str(row[IMAGE_COL])}
        if self.has_labels:
            dist = float(row[DISTANCE_COL])
            item.update({'object_target': torch.tensor(object_to_idx[str(row[OBJECT_COL])], dtype=torch.long), 'zone_target': torch.tensor(zone_to_idx[str(row[ZONE_COL])], dtype=torch.long), 'illumination_target': torch.tensor(illum_to_idx[str(row[ILLUM_COL])], dtype=torch.long), 'distance_raw': torch.tensor(dist, dtype=torch.float32), 'distance_target': torch.tensor((dist - dist_mean) / dist_std if CFG.USE_DISTANCE_NORMALIZATION else dist, dtype=torch.float32)})
        return item

train_ds = UltrasonicDataset(train_df, train_tf, True)
valid_ds = UltrasonicDataset(valid_df, valid_tf, True)
test_ds = UltrasonicDataset(test_df, valid_tf, False) if len(test_df) else None
sample = train_ds[0]
print({k: tuple(v.shape) if torch.is_tensor(v) else v for k, v in sample.items()})


## 6. Task-Aware Residual Model

Distance receives fused + audio features, object/illumination receive fused + image features, and zone receives fused features. This matches the assumption that ultrasonic reflections carry distance/depth while image carries appearance and illumination.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, c_in: int, c_out: int, stride: int = 1) -> None:
        super().__init__()
        self.main = nn.Sequential(nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.SiLU(inplace=True), nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out))
        self.skip = nn.Identity() if c_in == c_out and stride == 1 else nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))
        self.act = nn.SiLU(inplace=True)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.main(x) + self.skip(x))

class Encoder(nn.Module):
    def __init__(self, in_ch: int, embed_dim: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_ch, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True), ResidualBlock(32, 32), ResidualBlock(32, 32), ResidualBlock(32, 64, 2), ResidualBlock(64, 64), ResidualBlock(64, 128, 2), ResidualBlock(128, 128), ResidualBlock(128, 256, 2), ResidualBlock(256, 256), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim), nn.SiLU(inplace=True), nn.Dropout(dropout))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class GatedFusion(nn.Module):
    def __init__(self, dim_i: int, dim_a: int, out_dim: int) -> None:
        super().__init__()
        self.ip = nn.Linear(dim_i, out_dim)
        self.ap = nn.Linear(dim_a, out_dim)
        self.gate = nn.Sequential(nn.Linear(dim_i + dim_a, out_dim), nn.Sigmoid())
        self.out = nn.Sequential(nn.LayerNorm(out_dim), nn.SiLU(inplace=True))
    def forward(self, image_emb: torch.Tensor, audio_emb: torch.Tensor) -> torch.Tensor:
        gate = self.gate(torch.cat([image_emb, audio_emb], 1))
        return self.out(gate * self.ip(image_emb) + (1 - gate) * self.ap(audio_emb))

def mlp(in_dim: int, out_dim: int) -> nn.Sequential:
    return nn.Sequential(nn.Linear(in_dim, 128), nn.SiLU(inplace=True), nn.Dropout(CFG.DROPOUT), nn.Linear(128, out_dim))

class TaskAwareModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        if not CFG.USE_IMAGE and not CFG.USE_AUDIO:
            raise ValueError('enable at least one modality')
        self.image_encoder = Encoder(3, CFG.IMAGE_EMBED_DIM, CFG.DROPOUT) if CFG.USE_IMAGE else None
        self.audio_encoder = Encoder(1, CFG.AUDIO_EMBED_DIM, CFG.DROPOUT) if CFG.USE_AUDIO else None
        if CFG.USE_IMAGE and CFG.USE_AUDIO:
            self.fusion = GatedFusion(CFG.IMAGE_EMBED_DIM, CFG.AUDIO_EMBED_DIM, CFG.FUSION_DIM) if CFG.FUSION_TYPE == 'gated' else nn.Sequential(nn.Linear(CFG.IMAGE_EMBED_DIM + CFG.AUDIO_EMBED_DIM, CFG.FUSION_DIM), nn.LayerNorm(CFG.FUSION_DIM), nn.SiLU(inplace=True), nn.Dropout(CFG.DROPOUT))
            object_dim = CFG.FUSION_DIM + CFG.IMAGE_EMBED_DIM
            distance_dim = CFG.FUSION_DIM + CFG.AUDIO_EMBED_DIM
            zone_dim = CFG.FUSION_DIM
            illum_dim = CFG.FUSION_DIM + CFG.IMAGE_EMBED_DIM
        elif CFG.USE_IMAGE:
            object_dim = distance_dim = zone_dim = illum_dim = CFG.IMAGE_EMBED_DIM
        else:
            object_dim = distance_dim = zone_dim = illum_dim = CFG.AUDIO_EMBED_DIM
        self.object_head = mlp(object_dim, len(object_to_idx))
        self.distance_head = mlp(distance_dim, 1)
        self.zone_head = mlp(zone_dim, len(zone_to_idx))
        self.illum_head = mlp(illum_dim, len(illum_to_idx))
    def forward(self, image: torch.Tensor, spec: torch.Tensor) -> dict[str, torch.Tensor]:
        image_emb = self.image_encoder(image) if self.image_encoder is not None else None
        audio_emb = self.audio_encoder(spec) if self.audio_encoder is not None else None
        if image_emb is not None and audio_emb is not None:
            fused = self.fusion(image_emb, audio_emb) if CFG.FUSION_TYPE == 'gated' else self.fusion(torch.cat([image_emb, audio_emb], 1))
            object_feat = torch.cat([fused, image_emb], 1)
            distance_feat = torch.cat([fused, audio_emb], 1)
            zone_feat = fused
            illum_feat = torch.cat([fused, image_emb], 1)
        else:
            base = image_emb if image_emb is not None else audio_emb
            object_feat = distance_feat = zone_feat = illum_feat = base
        return {'object_logits': self.object_head(object_feat), 'distance_pred': self.distance_head(distance_feat).squeeze(-1), 'zone_logits': self.zone_head(zone_feat), 'illumination_logits': self.illum_head(illum_feat)}

model = TaskAwareModel().to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('trainable params:', f'{params:,}', 'size MB:', round(params * 4 / 1024**2, 2))


## 7. Loss and Competition Metric

Backpropagation uses the same standard weighted multi-task objective as the other files: CrossEntropyLoss for the three classification tasks and SmoothL1/MSE for distance.

In [ ]:
def maybe_class_weights(series: pd.Series, encoder: dict[str, int]) -> torch.Tensor | None:
    if not CFG.USE_CLASS_WEIGHTS:
        return None
    y = torch.tensor([encoder[str(v)] for v in series], dtype=torch.long)
    counts = torch.bincount(y, minlength=len(encoder)).float().clamp_min(1)
    return (counts.sum() / (len(encoder) * counts)).to(DEVICE)

losses = {'object': nn.CrossEntropyLoss(weight=maybe_class_weights(train_df[OBJECT_COL], object_to_idx)), 'zone': nn.CrossEntropyLoss(weight=maybe_class_weights(train_df[ZONE_COL], zone_to_idx)), 'illum': nn.CrossEntropyLoss(weight=maybe_class_weights(train_df[ILLUM_COL], illum_to_idx)), 'distance': nn.SmoothL1Loss() if CFG.DISTANCE_LOSS == 'smooth_l1' else nn.MSELoss()}

def multitask_loss(out: dict[str, torch.Tensor], batch: dict[str, torch.Tensor]) -> tuple[torch.Tensor, dict[str, float]]:
    obj = losses['object'](out['object_logits'], batch['object_target'])
    dist = losses['distance'](out['distance_pred'], batch['distance_target'])
    zone = losses['zone'](out['zone_logits'], batch['zone_target'])
    illum = losses['illum'](out['illumination_logits'], batch['illumination_target'])
    total = 0.40 * obj + 0.30 * dist + 0.20 * zone + 0.10 * illum
    return total, {'total_loss': float(total.detach().cpu()), 'object_loss': float(obj.detach().cpu()), 'distance_loss': float(dist.detach().cpu()), 'zone_loss': float(zone.detach().cpu()), 'illumination_loss': float(illum.detach().cpu())}

def denorm_distance(x: np.ndarray) -> np.ndarray:
    return x.astype(np.float32) * np.float32(dist_std) + np.float32(dist_mean) if CFG.USE_DISTANCE_NORMALIZATION else x.astype(np.float32)

def competition(y_true: dict[str, np.ndarray], y_pred: dict[str, np.ndarray]) -> dict[str, float]:
    pred_dist = denorm_distance(y_pred['distance'])
    rmse = float(np.sqrt(mean_squared_error(y_true['distance_raw'], pred_dist)))
    dscore = max(0.0, 1.0 - rmse / 2.0)
    of1 = float(f1_score(y_true['object'], y_pred['object'], average='macro', zero_division=0))
    zf1 = float(f1_score(y_true['zone'], y_pred['zone'], average='macro', zero_division=0))
    if1 = float(f1_score(y_true['illumination'], y_pred['illumination'], average='macro', zero_division=0))
    score = 100 * (0.40 * of1 + 0.30 * dscore + 0.20 * zf1 + 0.10 * if1)
    return {'object_macro_f1': of1, 'distance_rmse': rmse, 'distance_component': dscore, 'zone_macro_f1': zf1, 'illumination_macro_f1': if1, 'competition_score': score}


## 8. Train, Validate, and Save Best by Competition Score

In [ ]:
def loader(ds: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=shuffle, num_workers=CFG.NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', persistent_workers=CFG.NUM_WORKERS > 0)
train_loader = loader(train_ds, True)
valid_loader = loader(valid_ds, False)

def to_device(batch: dict[str, Any]) -> dict[str, Any]:
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

optimizer = AdamW(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=CFG.WEIGHT_DECAY)
total_steps = max(1, len(train_loader) * CFG.EPOCHS)
warmup = min(total_steps - 1, len(train_loader) * CFG.WARMUP_EPOCHS)
scheduler = LambdaLR(optimizer, lambda step: (step + 1) / warmup if warmup > 0 and step < warmup else 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warmup) / max(1, total_steps - warmup)))))
scaler = GradScaler(enabled=CFG.USE_AMP and DEVICE.type == 'cuda')

def train_epoch(epoch: int) -> dict[str, float]:
    model.train(); totals = {}; seen = 0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(tqdm(train_loader, desc=f'train {epoch}', leave=False), 1):
        batch = to_device(batch)
        with autocast(enabled=CFG.USE_AMP and DEVICE.type == 'cuda'):
            loss, logs = multitask_loss(model(batch['image'], batch['spec']), batch)
            loss = loss / CFG.GRADIENT_ACCUMULATION_STEPS
        scaler.scale(loss).backward()
        if step % CFG.GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        bs = batch['image'].size(0); seen += bs
        for k, v in logs.items(): totals[k] = totals.get(k, 0.0) + v * bs
    return {f'train_{k}': v / max(1, seen) for k, v in totals.items()}

@torch.no_grad()
def valid_epoch() -> tuple[dict[str, float], dict[str, np.ndarray], dict[str, np.ndarray]]:
    model.eval(); totals = {}; seen = 0
    yt = {'object': [], 'zone': [], 'illumination': [], 'distance_raw': []}; yp = {'object': [], 'zone': [], 'illumination': [], 'distance': []}
    for batch in tqdm(valid_loader, desc='valid', leave=False):
        batch = to_device(batch)
        with autocast(enabled=CFG.USE_AMP and DEVICE.type == 'cuda'):
            out = model(batch['image'], batch['spec']); loss, logs = multitask_loss(out, batch)
        bs = batch['image'].size(0); seen += bs
        for k, v in logs.items(): totals[k] = totals.get(k, 0.0) + v * bs
        yt['object'].append(batch['object_target'].cpu().numpy()); yp['object'].append(out['object_logits'].argmax(1).cpu().numpy())
        yt['zone'].append(batch['zone_target'].cpu().numpy()); yp['zone'].append(out['zone_logits'].argmax(1).cpu().numpy())
        yt['illumination'].append(batch['illumination_target'].cpu().numpy()); yp['illumination'].append(out['illumination_logits'].argmax(1).cpu().numpy())
        yt['distance_raw'].append(batch['distance_raw'].cpu().numpy()); yp['distance'].append(out['distance_pred'].cpu().numpy())
    y_true = {k: np.concatenate(v) for k, v in yt.items()}; y_pred = {k: np.concatenate(v) for k, v in yp.items()}
    return {**{f'val_{k}': v / max(1, seen) for k, v in totals.items()}, **competition(y_true, y_pred)}, y_true, y_pred

def save_ckpt(path: Path, epoch: int, best: float, history: list[dict[str, Any]]) -> None:
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'epoch': epoch, 'best_score': best, 'cfg': asdict(CFG), 'encoders': {'object': object_to_idx, 'zone': zone_to_idx, 'illumination': illum_to_idx}, 'decoders': {'object': idx_to_object, 'zone': idx_to_zone, 'illumination': idx_to_illum}, 'distance': {'mean': dist_mean, 'std': dist_std, 'normalized': CFG.USE_DISTANCE_NORMALIZATION}, 'history': history}, path)

history = []; best = -float('inf'); patience = 0; best_true = best_pred = None
for epoch in range(1, CFG.EPOCHS + 1):
    tr = train_epoch(epoch); va, y_true, y_pred = valid_epoch(); row = {'epoch': epoch, 'lr': optimizer.param_groups[0]['lr'], **tr, **va}; history.append(row)
    pd.DataFrame(history).to_csv(CFG.CHECKPOINT_DIR / 'history.csv', index=False)
    print(f"epoch {epoch:03d} | train {row['train_total_loss']:.4f} | val {row['val_total_loss']:.4f} | obj {row['object_macro_f1']:.4f} | zone {row['zone_macro_f1']:.4f} | illum {row['illumination_macro_f1']:.4f} | rmse {row['distance_rmse']:.4f} | score {row['competition_score']:.4f}")
    save_ckpt(CFG.CHECKPOINT_DIR / 'last_model.pt', epoch, max(best, row['competition_score']), history)
    if row['competition_score'] > best:
        best = row['competition_score']; patience = 0; best_true = y_true; best_pred = y_pred; save_ckpt(CFG.CHECKPOINT_DIR / 'best_model.pt', epoch, best, history); print('  saved best')
    else:
        patience += 1
        if patience >= CFG.EARLY_STOPPING_PATIENCE:
            print('early stopping'); break
print('best score:', best)


## 9. Validation Analysis and Submission

In [ ]:
hist = pd.DataFrame(history)
if len(hist):
    for cols, title in [(['train_total_loss', 'val_total_loss'], 'loss'), (['competition_score'], 'score'), (['distance_rmse'], 'distance rmse'), (['object_macro_f1', 'zone_macro_f1', 'illumination_macro_f1'], 'macro f1')]:
        hist.plot(x='epoch', y=cols, figsize=(8, 4), marker='o'); plt.title(title); plt.grid(True); plt.show()

if best_true is not None:
    for task, labels in [('object', idx_to_object), ('zone', idx_to_zone), ('illumination', idx_to_illum)]:
        cm = confusion_matrix(best_true[task], best_pred[task], labels=list(labels.keys()))
        ConfusionMatrixDisplay(cm, display_labels=[labels[i] for i in labels]).plot(xticks_rotation=45); plt.title(task); plt.show()
        p, r, f, s = precision_recall_fscore_support(best_true[task], best_pred[task], labels=list(labels.keys()), zero_division=0)
        display(pd.DataFrame({'task': task, 'class': [labels[i] for i in labels], 'precision': p, 'recall': r, 'f1': f, 'support': s}))
    pred_d = denorm_distance(best_pred['distance']); true_d = best_true['distance_raw']
    plt.scatter(true_d, pred_d, s=10); plt.xlabel('true distance'); plt.ylabel('predicted distance'); plt.grid(True); plt.show()
    plt.hist(pred_d - true_d, bins=30); plt.title('distance residuals'); plt.show()

@torch.no_grad()
def create_submission() -> pd.DataFrame:
    if test_ds is None:
        raise ValueError('No test set loaded')
    ckpt = torch.load(CFG.CHECKPOINT_DIR / 'best_model.pt', map_location=DEVICE)
    model.load_state_dict(ckpt['model']); model.eval()
    dl = loader(test_ds, False)
    rows = []
    for batch in tqdm(dl, desc='predict'):
        batch = to_device(batch); out = model(batch['image'], batch['spec'])
        obj = out['object_logits'].argmax(1).cpu().numpy(); zone = out['zone_logits'].argmax(1).cpu().numpy(); illum = out['illumination_logits'].argmax(1).cpu().numpy(); dist = denorm_distance(out['distance_pred'].cpu().numpy())
        for sid, o, z, il, d in zip(batch['sample_id'], obj, zone, illum, dist):
            rows.append({IMAGE_COL: sid, OBJECT_COL: idx_to_object[int(o)], DISTANCE_COL: float(d), ZONE_COL: idx_to_zone[int(z)], ILLUM_COL: idx_to_illum[int(il)]})
    sub = pd.DataFrame(rows); sub.to_csv(CFG.SUBMISSION_PATH, index=False); print(sub.shape); display(sub.head()); print('saved:', CFG.SUBMISSION_PATH); return sub

if test_ds is not None and (CFG.CHECKPOINT_DIR / 'best_model.pt').exists():
    submission = create_submission()
